# Dostrajanie modelu LLM do klasyfikacji zwrotek polskich utworów rapowych na ich wykonawców.


## Instalacja, import bibliotek i patchowanie


In [1]:
import os

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import TaskType
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"

# Unsloth przed transformers (optymalizacje + patch Gemma4)
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

from transformers import (
    AutoModel,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_layers import GenericForSequenceClassification
from transformers.models.auto.modeling_auto import (
    MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING,
)
from transformers.models.gemma4.configuration_gemma4 import Gemma4Config
from transformers.models.gemma4.modeling_gemma4 import Gemma4PreTrainedModel

# Workaround: patch Unsloth ukrywa num_kv_shared_layers==0 i psuje AutoConfig.__repr__
_orig_to_diff_dict = PretrainedConfig.to_diff_dict

def _safe_to_diff_dict(self):
    if "Gemma4" in self.__class__.__name__:
        return self.to_dict()
    return _orig_to_diff_dict(self)


PretrainedConfig.to_diff_dict = _safe_to_diff_dict

# Transformers 5.10 nie ma jeszcze gotowej klasy dla Gemma 4 - rejestrujemy ją ręcznie.
class Gemma4ForSequenceClassification(GenericForSequenceClassification, Gemma4PreTrainedModel):
    def __init__(self, config):
        Gemma4PreTrainedModel.__init__(self, config)
        self.num_labels = config.num_labels
        setattr(self, self.base_model_prefix, AutoModel.from_config(config))
        self.score = torch.nn.Linear(
            config.get_text_config().hidden_size, self.num_labels, bias=False
        )
        self.post_init()


MODEL_FOR_SEQUENCE_CLASSIFICATION_MAPPING.register(
    Gemma4Config, Gemma4ForSequenceClassification, exist_ok=True
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Wczytanie datasetu


In [2]:
DATASET_DIR = "dataset"

train_df = pd.read_json(os.path.join(DATASET_DIR, "train.json"))
test_df = pd.read_json(os.path.join(DATASET_DIR, "test.json"))

print(f"Zbiór treningowy: {len(train_df)} zwrotek")
print(f"Zbiór testowy:    {len(test_df)} zwrotek")
print(f"Liczba unikalnych etykiet ze zbioru treningowego: {train_df['label'].nunique()}")
train_df.head(3)

Zbiór treningowy: 6815 zwrotek
Zbiór testowy:    1203 zwrotek
Liczba unikalnych etykiet ze zbioru treningowego: 20


,song,section,text,label
0,Chcę żyć,zwrotka,Zawsze byłem i będę twardy\nJak każdy myśli o ...,Sobel
1,Kapie,zwrotka,Śmieszy mnie to co nazywacie trapem\nKilka but...,Kabe
2,Frank Lentini,intro,- Kurwa już trzeci raz ci się nagrywam na tą p...,Diho


## Kodowanie etykiet


In [3]:
labels = sorted(train_df["label"].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
NUM_LABELS = len(labels)

# Usunięcie ze zbioru testowego zwrotek o nieznanych etykietach (niewystępujących w zbiorze treningowym)
test_df = test_df[test_df["label"].isin(label2id)].reset_index(drop=True)

# Mapowanie etykiet na indeksy
train_df = train_df.copy()
train_df["labels"] = train_df["label"].map(label2id)
test_df["labels"] = test_df["label"].map(label2id)

# Wydzielenie zbioru walidacyjnego ze zbioru treningowego
train_df, val_df = train_test_split(
    train_df,
    test_size=0.10,
    random_state=1111,
    stratify=train_df["labels"],
)

print(f"Liczba klas (artystów): {NUM_LABELS}")
print(f"Trening: {len(train_df)}, Walidacja: {len(val_df)}, Testowanie: {len(test_df)}")

Liczba klas (artystów): 20
Trening: 6133, Walidacja: 682, Testowanie: 1203


## Wczytanie Gemma 4 E2B przez Unsloth


In [4]:
MODEL_NAME = "unsloth/gemma-4-E2B"
MAX_LENGTH = 650  # ok. 99. percentyl

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME, # Nazwa modelu do pobrania z HuggingFace
    max_seq_length=MAX_LENGTH, # Maksymalna ilość tokenów na wejściu dla modelu
    load_in_4bit=False,
    load_in_16bit=True,
    dtype=None,
    auto_model=AutoModelForSequenceClassification, # Model do klasyfikacji sekwencji (z głowicą klasyfikacyjną)
    num_labels=NUM_LABELS, # Liczba klas (artystów do klasyfikacji)
    id2label=id2label, # Mapowanie indeksów na etykiety
    label2id=label2id, # Mapowanie etykiet na indeksy
    trust_remote_code=False,
    ignore_mismatched_sizes=True,
    use_exact_model_name=True, # Wymuszenie użycia dokładnej nazwy modelu (bez automatycznego dopasowania do innej wersji)
    device_map={"": 0}, # Wymuszenie użycia GPU 0
)

# Konfiguracja tokenizera dla modelu Gemma 4 (szablon konwersacyjny)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TOKENIZER_CACHE = "gemma4_tokenizer_cache"
tokenizer.save_pretrained(TOKENIZER_CACHE)
tokenizer.name_or_path = TOKENIZER_CACHE

# Konfiguracja adapterów LoRA dla modelu Gemma 4 (z wyłączonym trenowaniem warstw wizyjnych)
model = FastModel.get_peft_model(
    model,
    r=32, # Rząd macierzy LoRA
    lora_alpha=64,
    lora_dropout=0.0, # Musi być równe 0, aby optymalizacje unsloth zadziałały dla macierzy LoRA
    bias="none",
    task_type=TaskType.SEQ_CLS, # Typ zadania ustawiony na klasyfikację sekwencji
    modules_to_save=["score"], # Zapisanie głowicy klasyfikacyjnej (score) wraz z adapterami LoRA
    finetune_vision_layers=False, # Wyłączenie trenowania warstw wizyjnych
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    use_gradient_checkpointing=False,
    random_state=1111,
)

# Wymuszenie gradientów wejściowych dla adapterów LoRA
model.enable_input_require_grads()

text_config = model.config.get_text_config()
text_config.pad_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

model.print_trainable_parameters()

==((====))==  Unsloth 2026.6.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H200. Num GPUs = 1. Max memory: 139.801 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Gemma4ForSequenceClassification LOAD REPORT from: unsloth/gemma-4-E2B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Unsloth: Restored added_tokens_decoder metadata in gemma4_tokenizer_cache/tokenizer_config.json.


Unsloth: Allowing gradients for `base_model.model.score` since it's in `modules_to_save`.
trainable params: 50,706,432 || all params: 5,173,915,168 || trainable%: 0.9800


## Formatowanie wejścia i tokenizacja datasetów


In [5]:
# Prompt do klasyfikacji (bo Gemma 4 jest konwersacyjna)
CLASSIFICATION_PROMPT = """Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
{verse}"""

# Funkcja formatująca zwrotkę do postaci promptu dla modelu konwersacyjnego
def format_input(verse: str) -> str:
    messages = [{"role": "user", "content": CLASSIFICATION_PROMPT.format(verse=verse)}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return text.removeprefix("<bos>")

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# Przygotowanie datasetów HuggingFace z formatowaniem tekstu i tokenizacją
keep_cols = ["text", "labels"]
for df in (train_df, val_df, test_df):
    df["verse"] = df["text"]
    df["text"] = df["verse"].map(format_input)

# Konwersja DataFrame'ów na datasety HuggingFace i tokenizacja
train_ds = Dataset.from_pandas(train_df[keep_cols], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[keep_cols], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[keep_cols], preserve_index=False)

# Tokenizacja datasetów (z usunięciem kolumny "text", bo po tokenizacji nie jest już potrzebna)
train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_ds = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds = test_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

# Data collator do dynamicznego paddingu podczas treningu
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Przykład wejścia:\n", train_df.iloc[0]["text"][:400], "\n...")
print(train_ds)

Map:   0%|          | 0/6133 [00:00<?, ? examples/s]

Map:   0%|          | 0/682 [00:00<?, ? examples/s]

Map:   0%|          | 0/1203 [00:00<?, ? examples/s]

Przykład wejścia:
 <|turn>user
Przeczytaj poniższą zwrotkę polskiego utworu rapowego i określ, kto jest wykonawcą.

Tekst zwrotki:
Co o nas mówią w komentach to zgrali
Przeszedłem w inny wymiar, my jesteśmy kosmitami
Chciałbym się z tobą bratać
Nie jesteśmy tacy sami
Nawet jak spale 5 i tu przysypiam
To tylko twój skład tak lami
Co o nas mówią w komentach to zgrali
Przeszedłem w inny wymiar, my jesteśmy kosmitami
Ch 
...
Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 6133
})


## Definicja metryk, wagi klas i label smoothing


In [6]:
# Współczynnik label smoothing dla funkcji straty
LABEL_SMOOTHING = 0.1

# Obliczanie wag klas dla funkcji straty (klasy mniej liczne dostają większą wagę, co pomaga modelowi lepiej uczyć się z niezbalansowanych danych)
label_counts = train_df["labels"].value_counts()
class_weights = torch.tensor(
    [len(train_df) / (NUM_LABELS * label_counts[i]) for i in range(NUM_LABELS)],
    dtype=torch.float32,
)

# Funkcja obliczająca metryki do ewaluacji (accuracy, macro F1, top-3 accuracy)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    top3 = np.argsort(logits, axis=-1)[:, -3:]
    top3_acc = np.mean([labels[i] in top3[i] for i in range(len(labels))])
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "top3_accuracy": top3_acc,
    }

## Definicja customowej klasy Trainer


In [ ]:
from contextlib import contextmanager

# Custom Trainer, który uwzględnia wagi klas, label smoothing i osobne LR dla LoRA i głowicy
class WeightedTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        label_smoothing=0.0,
        lora_lr=3e-4,
        head_lr=1e-3,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.label_smoothing = label_smoothing
        self.lora_lr = lora_lr
        self.head_lr = head_lr

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer

        head_params, lora_params = [], []
        for name, param in self.model.named_parameters():
            if not param.requires_grad:
                continue
            (head_params if "score" in name else lora_params).append(param)

        optimizer_grouped_parameters = [
            {
                "params": lora_params,
                "lr": self.lora_lr,
                "weight_decay": self.args.weight_decay,
            },
            {
                "params": head_params,
                "lr": self.head_lr,
                "weight_decay": self.args.weight_decay,
            },
        ]
        optimizer_cls, optimizer_kwargs = Trainer.get_optimizer_cls_and_kwargs(self.args)
        optimizer_kwargs.pop("lr", None)
        self.optimizer = optimizer_cls(optimizer_grouped_parameters, **optimizer_kwargs)
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(
            logits,
            labels,
            weight=weight,
            label_smoothing=self.label_smoothing,
        )
        return (loss, outputs) if return_outputs else loss

    @contextmanager
    def _pause_notebook_progress(self):
        from transformers.utils.notebook import NotebookProgressCallback

        removed = [
            cb
            for cb in list(self.callback_handler.callbacks)
            if isinstance(cb, NotebookProgressCallback)
        ]
        for cb in removed:
            self.remove_callback(cb)
        try:
            yield
        finally:
            for cb in removed:
                self.add_callback(cb)

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        with self._pause_notebook_progress():
            return super().evaluate(
                eval_dataset=eval_dataset,
                ignore_keys=ignore_keys,
                metric_key_prefix=metric_key_prefix,
            )

    def predict(self, test_dataset, ignore_keys=None, metric_key_prefix="test"):
        with self._pause_notebook_progress():
            return super().predict(
                test_dataset,
                ignore_keys=ignore_keys,
                metric_key_prefix=metric_key_prefix,
            )

## Konfiguracja parametrów treningu


In [ ]:
LORA_LR = 2e-4  # Adaptery LoRA
HEAD_LR = 1e-3  # Głowica klasyfikacyjna

OUTPUT_DIR = "outputs-gemma4-rap-classifier-unsloth"

use_cuda = torch.cuda.is_available()
dataloader_num_workers = 0 if os.name == "nt" else 4

# Konfiguracja treningu
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=12, # Liczba epok treningu
    per_device_train_batch_size=32, # Batch size podczas treningu
    per_device_eval_batch_size=64, # Batch size podczas ewaluacji
    gradient_accumulation_steps=2,
    learning_rate=LORA_LR, # bazowy LR
    weight_decay=0.01, # Parametr regularyzacji
    warmup_ratio=0.1, # Procent kroków rozgrzewkowych
    lr_scheduler_type="linear", # Liniowy scheduler dla tempa uczenia
    optim="adamw_torch_fused", # Optymalizator
    eval_strategy="epoch", # Ewaluacja po każdej epoce
    save_strategy="epoch", # Zapisywanie modelu po każdej epoce
    save_total_limit=1,
    load_best_model_at_end=True, # Ładowanie najlepszego modelu na koniec treningu
    metric_for_best_model="eval_f1_macro", # Metryka do wyboru najlepszego modelu
    greater_is_better=True, # W przypadku F1 większa wartość jest lepsza
    label_names=["labels"],
    train_sampling_strategy="group_by_length",
    dataloader_num_workers=dataloader_num_workers,
    dataloader_pin_memory=True,
    tf32=use_cuda and torch.cuda.is_tf32_supported(),
    logging_steps=10,
    seed=1111,
    bf16=use_cuda and torch.cuda.is_bf16_supported(),
    fp16=use_cuda and not torch.cuda.is_bf16_supported(),
    report_to="none",
)

# Inicjalizacja customowego trenera z uwzględnieniem wag klas i label smoothing
trainer = WeightedTrainer(
    model=model, # Model do trenowania
    args=training_args, # Konfiguracja treningu
    train_dataset=train_ds, # Dataset treningowy
    eval_dataset=val_ds, # Dataset walidacyjny
    processing_class=tokenizer, # Przetwarzanie danych przez tokenizer
    data_collator=data_collator, # Dynamiczny padding
    compute_metrics=compute_metrics, # Funkcja obliczająca metryki do ewaluacji
    class_weights=class_weights, # Wagi klas dla funkcji straty
    label_smoothing=LABEL_SMOOTHING, # Współczynnik label smoothing dla funkcji straty
    lora_lr=LORA_LR, # Learning rate dla adapterów LoRA
    head_lr=HEAD_LR, # Learning rate dla głowicy klasyfikacyjnej
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)], # Wczesne zatrzymanie treningu, jeśli metryka ewaluacji nie poprawi się przez 2 kolejne epoki
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Trening modelu


In [9]:
import shutil
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

train_result = trainer.train(resume_from_checkpoint=False)

print(f"\nCzas treningu: {train_result.metrics['train_runtime']:.0f} s")
print(f"Końcowy loss: {train_result.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,133 | Num Epochs = 12 | Total steps = 1,152
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 50,706,432 of 5,173,915,168 (0.98% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-96/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-192/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-288/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-384/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-480/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-576/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-672/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs-gemma4-rap-classifier-unsloth/checkpoint-768/tokenize


Czas treningu: 656 s
Końcowy loss: 1.6696


## Ewaluacja na zbiorze testowym


In [10]:
test_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix="test")
print(f"Accuracy: {test_metrics['test_accuracy']:.2%}")
print(f"Macro-F1: {test_metrics['test_f1_macro']:.4f}")
print(f"Top-3 accuracy: {test_metrics['test_top3_accuracy']:.2%}")

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Accuracy: 56.86%
Macro-F1: 0.6000
Top-3 accuracy: 70.82%


### Sprawdzenie dokładności w zależności od wykonawcy


In [ ]:
predictions = trainer.predict(test_ds)
pred_ids = np.argmax(predictions.predictions, axis=-1)
true_ids = predictions.label_ids

results_df = pd.DataFrame({
    "label": [id2label[i] for i in true_ids],
    "prediction": [id2label[i] for i in pred_ids],
})
results_df["correct"] = results_df["label"] == results_df["prediction"]

print("Dokładność w zależności od wykonawcy:")
per_label = (
    results_df.groupby("label")["correct"]
    .agg(["mean", "count"])
    .sort_values("count", ascending=False)
)
print(per_label.to_string())

Dokładność w zależności od wykonawcy:
                   mean  count
label                         
Sentino        0.093023    129
Young Igi      0.045455    110
Zeamsone       0.022727     88
Quebonafide    0.022727     88
White 2115     0.034483     87
Malik Montana  0.012048     83
Pezet          0.036585     82
Bedoes         0.025000     80
Żabson         0.042857     70
Sobel          0.014925     67
Mata           0.031250     64
OKI            0.037037     54
Kabe           0.106383     47
Diho           0.000000     44
Guzior         0.062500     32
Avi            0.076923     26
Young Leosia   0.000000     23
Oki            0.000000     16
OG Olgierd     0.250000      8
Louis Villain  0.200000      5


## Predykcja dla pojedynczych zwrotek


In [12]:
@torch.no_grad()
def predict_artist(verse: str, top_k: int = 3):
    model.for_inference()
    inputs = tokenizer(
        format_input(verse),
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(model.device)
    probs = model(**inputs).logits.softmax(-1)[0]
    top = probs.topk(min(top_k, NUM_LABELS))
    return [(id2label[i.item()], p.item()) for p, i in zip(top.values, top.indices)]


samples = test_df.sample(5, random_state=1111)
for _, row in samples.iterrows():
    ranking = predict_artist(row["verse"])
    top_label = ranking[0][0]
    status = "TRAFIONO" if top_label == row["label"] else "POMYŁKA"
    print(f"[{status}] utwór: {row['song']}")
    print(f"Zwrotka: {' / '.join(row['verse'].splitlines())}")
    print(f"Prawda: {row['label']}")
    print(f"Predykcja: " + ", ".join(f"{a} ({p:.0%})" for a, p in ranking) + "\n")

[TRAFIONO] utwór: Taki lajf 2
Zwrotka: Stoję pod kałachem pod domem twojej starej / Biję się jak dziecko, ale strzelam tak jak Paryż / Będziesz jak kaleka łaził z workiem na szczyny / Moi ludzie z Nador cię wycisną jak cytryny / Jaka to bajera, że ty grozisz przez komórkę? / Już na samym dołku cię uduszą tutaj sznurkiem / Sprzedaję dzieciom dragi, płyty, czekolady / Dostaniesz śrubokrętem w nerę, pozdro z Arabii / 81 Gang Life, Sicarios całe bloody / Trzy Viagry, cztery kurwy na wakacjach w Bali / Walther w Messengerze, żeby przekazać wiadomość / Można obciąć rękę, a ty będziesz miał świadomość / Małe pizdy huczą coś na necie, lepiej chodź już / Paralizatorem kurwie w serce i na okup / Całe życie w getcie, w całej duszy lód / Jestem z trzeciego świata, kurwo, nie wiesz co to brud
Prawda: Sentino
Predykcja: Sentino (54%), Louis Villain (12%), OG Olgierd (8%)

[POMYŁKA] utwór: Powiedz co
Zwrotka: Czy się chcę wyleczyć? / Szukam miłości czy się chcę wyleczyć? / Czy się chcę wyleczyć / Szu

## Zapis adapterów LoRA i głowicy klasyfikacyjnej


In [13]:
SAVE_DIR = "gemma4-e2b-polish-rap-classifier"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Adaptery LoRA i klasyfikator zapisane w: {SAVE_DIR}")

Unsloth: Restored added_tokens_decoder metadata in gemma4-e2b-polish-rap-classifier/tokenizer_config.json.


Adaptery LoRA i klasyfikator zapisane w: gemma4-e2b-polish-rap-classifier
